# Entity sync via import-dataset

Demonstrates **blocking** entity `.sync()` calls that stage PDB/SDF/CSV files and run one
hidden `deeporigin.import-dataset` execution per call.

**Local stack (platform-toolbox):**

1. Build the `import-dataset` image (`images/import-dataset`).
2. Start the toolbox test gateway (`tests/gateway`) with `local-dev-ports.json` so
   `deeporigin.import-dataset` routes to the served container (default host port **8060**).
3. Point `DeepOriginClient` at the gateway-backed platform URL for your org.

Requires a project (`client.project_id`).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from deeporigin import config
config.set_env("local")

In [ ]:
from deeporigin import projects
from deeporigin.drug_discovery import BRD_DATA_DIR, LigandSet, Pose, PoseSet, Protein
from deeporigin.platform import DeepOriginClient

projects.create("entity-sync-demo")
client = DeepOriginClient()
client

## SMILES-only ligands (CSV / `process_csv`)

Structure-less ligands are serialized to a SMILES CSV and imported in one execution.

In [ ]:
ligands = LigandSet.from_smiles(["CCO", "CCCO", "c1ccccc1"])
ligands

In [ ]:
ligands.sync()
ligands

In [ ]:
ligands[0]

## Protein (`process_pdb`)

Sync the bundled BRD structure.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync(client=client)
protein

## Poses from SDF (`process_sdf` + `register_poses`)

Each pose must have `protein_id` set before `PoseSet.sync()`.

In [ ]:
from deeporigin.drug_discovery import Ligand

ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
ligand.sync()
pose = Pose(
    ligand_id=ligand.id,
    local_path=str(BRD_DATA_DIR / "brd-2.sdf"),
    smiles=ligand.smiles,
    protein_id=protein.id,
    project_id=client.project_id,
)
pose.sync(client=client)
pose